# Fine-tune a fraud typology extractor with QLoRA (Colab / Kaggle T4)

1. Install and clone the repo
2. Get data (generate with an LLM, or upload your verified `test.jsonl`)
3. Train QLoRA adapter (~30-60 min on a T4 for a 3B model, 2 epochs)
4. Merge the adapter, serve with vLLM, evaluate against baselines
5. (Optional) push the model to the Hugging Face Hub

Runtime -> Change runtime type -> **T4 GPU**.

In [ ]:
!git clone https://github.com/Keerthana-2003/fraud-typology-extractor.git
%cd fraud-typology-extractor
!pip install -q -r requirements.txt -r requirements-train.txt && pip install -q -e .
!nvidia-smi

## 2. Data
Generate narratives with a strong LLM (needs an API key). Labels are sampled first, so they are known by construction.
If you already generated and hand-verified data locally, upload `data/train.jsonl`, `data/val.jsonl`, and `data/test.jsonl` instead and skip to step 3.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
!python -m typology.generate_data --writer llm --n 4000 --out data/raw.jsonl
!python -m typology.prepare_data split

**Hand-verify the test set.** Download `data/test_to_verify.csv`, check each row, mark `verified` = `y` (fix labels if needed),
upload it back, then import. Never train on these rows.

In [ ]:
# from google.colab import files; files.download("data/test_to_verify.csv")   # review locally
# uploaded = files.upload()                                                     # upload the reviewed CSV
!python -m typology.prepare_data import --csv data/test_to_verify.csv --out data/test.jsonl

## 3. Train

In [ ]:
!python -m typology.train_qlora --model Qwen/Qwen2.5-3B-Instruct --out outputs/qwen3b-qlora \
    --epochs 2 --rank 16 --alpha 32 --batch-size 4 --grad-accum 4 --max-length 1024

## 4. Merge, serve with vLLM, evaluate

In [ ]:
!python -m typology.merge_adapter --base Qwen/Qwen2.5-3B-Instruct --adapter outputs/qwen3b-qlora --out models/qwen3b-fraud-typology
!pip install -q -r requirements-serve-gpu.txt

In [ ]:
# Fine-tuned model on port 8001 and the base model on port 8002 (T4 has 16 GB: run them one at a time if memory is tight)
import subprocess, time, requests
def serve(model, name, port, mem="0.45"):
    p = subprocess.Popen(["vllm", "serve", model, "--served-model-name", name, "--port", str(port),
                          "--max-model-len", "2048", "--gpu-memory-utilization", mem, "--dtype", "half"],
                         stdout=open(f"vllm_{name}.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            if requests.get(f"http://localhost:{port}/v1/models").ok: return p
        except Exception: pass
        time.sleep(5)
    raise RuntimeError(open(f"vllm_{name}.log").read()[-2000:])
ft = serve("models/qwen3b-fraud-typology", "ft", 8001)

In [ ]:
GPU_COST = 0.35   # USD/hour for a T4 on a cloud provider; set to what you'd actually pay
!python -m typology.evaluate --name "Qwen2.5-3B QLoRA (ours)" --model ft --base-url http://localhost:8001/v1 --gpu-cost-per-hour {GPU_COST} --concurrency 32
ft.terminate(); time.sleep(10)
base = serve("Qwen/Qwen2.5-3B-Instruct", "base", 8002)
!python -m typology.evaluate --name "Qwen2.5-3B base (zero-shot)" --model base --base-url http://localhost:8002/v1 --gpu-cost-per-hour {GPU_COST} --concurrency 32
!python -m typology.evaluate --name "Qwen2.5-3B base (few-shot)" --model base --few-shot --base-url http://localhost:8002/v1 --gpu-cost-per-hour {GPU_COST} --concurrency 32
base.terminate()

In [ ]:
# Frontier API baseline (update prices to the current list price)
!python -m typology.evaluate --name "gpt-4o (few-shot)" --model gpt-4o --few-shot --base-url https://api.openai.com/v1 --price-in 2.50 --price-out 10.00 --concurrency 8
!cat results/results.md

## 5. Save results and (optionally) push the model

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# !python -m typology.merge_adapter --adapter outputs/qwen3b-qlora --out models/qwen3b-fraud-typology --push-to-hub YOUR_HF_USERNAME/qwen3b-fraud-typology
# from google.colab import files; files.download("results/results.md"); files.download("results/results.json")